In [1]:
import json
from pathlib import Path

def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan.")

_ROOT = cari_akar_repo()
_j = json.load(open(_ROOT / "data" / "raw" / "bps" / "2026-10-17" / "json" / "sdg8_tpt.json", encoding="utf-8"))
print(type(_j), len(_j))
if isinstance(_j, dict):
    print(list(_j.keys())[:10])
    for k, v in list(_j.items())[:3]:
        print(k, type(v), (list(v.keys())[:8] if isinstance(v, dict) else str(v)[:150]))
elif isinstance(_j, list):
    print(type(_j[0]), (list(_j[0].keys())[:10] if isinstance(_j[0], dict) else str(_j[0])[:150]))

<class 'list'> 14
<class 'dict'> ['status', 'data-availability', 'last_update', 'subject', 'var', 'turvar', 'labelvervar', 'vervar', 'tahun', 'turtahun']


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan. Buka notebook dari dalam folder repo.")


ROOT = cari_akar_repo()
RAW = ROOT / "data" / "raw"
PROFIL = ROOT / "reports" / "profil"
GAMBAR = ROOT / "reports" / "gambar"
PROFIL.mkdir(parents=True, exist_ok=True)
GAMBAR.mkdir(parents=True, exist_ok=True)

BERKAS_RAW = "bps/2026-10-17/sdg8_tpt.csv"
JUMLAH_CALON = 77            # hasil revisi #10

# ---- ambang yang dipakai (dicatat di log; ubah hanya dengan alasan tertulis) ----
MIN_PROV = 10                # minimal provinsi yang punya selisih pada tahun itu
SHARE_ARAH = 0.80            # minimal porsi provinsi yang bergerak searah
Z_ROBUST = 3.5               # ambang z-robust (MAD) untuk median selisih lintas provinsi

# ---- data ----
raw = pd.read_csv(RAW / BERKAS_RAW)
raw["nama"] = raw["wilayah"].astype(str).str.strip()
kode = pd.to_numeric(raw["wilayah_id"], errors="coerce")
raw["kode"] = np.where(kode < 100, kode * 100, kode).astype(int)
nasional = raw["nama"].str.contains("indonesia", case=False)
prov = raw[~nasional].copy()

cand = pd.read_csv(PROFIL / "daftar_calon_anomali.csv")
assert len(cand) == JUMLAH_CALON, f"calon {len(cand)} beda dari {JUMLAH_CALON}"
assert cand["id_calon"].is_unique and cand["indeks_raw"].notna().all()

# ============ 1. DUPLIKAT ============
KUNCI = ["wilayah_id", "turvar_id", "tahun", "periode"]
n_kunci_kembar = int(raw.duplicated(KUNCI, keep=False).sum())
n_baris_identik = int(raw.duplicated(keep=False).sum())
print(f"Duplikat: baris dengan kunci kembar = {n_kunci_kembar} | baris identik penuh = {n_baris_identik}")
print("Snapshot BPS di paket:", sorted(p.name for p in (RAW / "bps").iterdir() if p.is_dir()))

# ============ 2. PATAHAN SERENTAK ============
baris_stat = []
for per in ["Februari", "Agustus", "Tahunan"]:
    w = prov[prov["periode"] == per].pivot_table(index="tahun", columns="kode", values="nilai", aggfunc="first").sort_index()
    if len(w) < 3:
        continue
    selang = pd.Series(w.index, index=w.index).diff()
    d = w.diff()
    d[selang != 1] = np.nan                         # hanya tahun berurutan
    med = d.median(axis=1)
    n = d.notna().sum(axis=1)
    arah = np.sign(med)
    share = ((np.sign(d).mul(arah, axis=0)) > 0).sum(axis=1) / n.replace(0, np.nan)
    m0 = med.median()
    mad = (med - m0).abs().median()
    z = 0.6745 * (med - m0) / (mad if mad > 0 else np.nan)
    ser = (n >= MIN_PROV) & (share >= SHARE_ARAH) & (z.abs() >= Z_ROBUST)
    th = pd.Series(w.index, index=w.index)
    ok_next = (th.shift(-1) - th) == 1
    ok_prev = (th - th.shift(1)) == 1

    def lawan(a, b):                       # b membatalkan sebagian besar a (arah berlawanan)
        return (np.sign(a) != np.sign(b)) & ((a + b).abs() <= 0.5 * a.abs())

    balik_next = ser & ser.shift(-1, fill_value=False) & ok_next & lawan(med, med.shift(-1))
    balik_prev = ser & ser.shift(1, fill_value=False) & ok_prev & lawan(med.shift(1), med)
    stat = pd.DataFrame({"periode": per, "tahun": w.index, "n_prov": n.values, "median_selisih": med.values,
                         "share_searah": share.values, "z_robust": z.values, "serentak": ser.values,
                         "berbalik": (balik_next | balik_prev).values})
    baris_stat.append(stat)
stat = pd.concat(baris_stat, ignore_index=True)

# bukti metadata: tahun patahan yang dicatat ILO pada seri total 15+
ilo_break = {}
for nama in ["DF_SDG_0852_SEX_AGE_RT", "DF_UNE_DEAP_SEX_AGE_RT"]:
    d = pd.read_csv(RAW / "ilostat" / "2026-10-20" / "data" / f"{nama}.csv", dtype=str)
    d = d[(d["FREQ"] == "A") & (d["SEX"] == "SEX_T") & (d["AGE"] == "AGE_YTHADULT_YGE15")]
    ada = d["NOTE_INDICATOR"].fillna("").str.contains("Break in series")
    for t in d.loc[ada, "TIME_PERIOD"].str[:4].astype(int):
        ilo_break.setdefault(t, []).append(nama.replace("DF_", ""))
print("Tahun patahan menurut catatan ILO:", {k: v for k, v in sorted(ilo_break.items())})

stat["catatan_ilo_patahan"] = stat["tahun"].map(lambda t: ", ".join(ilo_break.get(t, [])))


def kategori_tahun(r):
    if not r["serentak"]:
        return ""
    if r["berbalik"]:
        return "guncangan_serentak_bukan_patahan"        # naik lalu berbalik: bukan perubahan definisi yang menetap
    return "patahan_seri" if r["catatan_ilo_patahan"] else "patahan_dugaan"


stat["kategori"] = stat.apply(kategori_tahun, axis=1)

# keputusan manual setelah membaca metadata (isi hanya dengan alasan tertulis di log)
OVERRIDE = {(2021, "Februari"): ("guncangan_serentak_bukan_patahan",
            "Agustus 2020 sudah naik serentak lalu turun bertahap di 2021-2023; bukan perubahan definisi yang menetap")}   # contoh: {(2013, "Februari"): ("patahan_seri", "alasan ...")}
for (t, p), (kat, alasan) in OVERRIDE.items():
    stat.loc[(stat["tahun"] == t) & (stat["periode"] == p), "kategori"] = kat

serentak = stat[stat["serentak"]].copy()
print("\nTahun-periode serentak:")
print(serentak[["periode", "tahun", "n_prov", "median_selisih", "share_searah", "z_robust", "berbalik", "catatan_ilo_patahan", "kategori"]].round(2).to_string(index=False)
      if len(serentak) else "  tidak ada")
patahan_set = set(map(tuple, stat.loc[stat["kategori"].isin(["patahan_seri", "patahan_dugaan"]), ["tahun", "periode"]].values.tolist()))
serentak_set = set(map(tuple, serentak[["tahun", "periode"]].values.tolist()))

# ============ 3. KLASIFIKASI 77 CALON ============
seri = {per: prov[prov["periode"] == per].pivot_table(index="tahun", columns="kode", values="nilai", aggfunc="first").sort_index()
        for per in prov["periode"].unique()}


def konteks(r):
    k, t, p, v = r["kode"], r["tahun"], r["periode"], r["nilai"]
    w = seri[p][k].dropna()
    tetangga = w[(w.index != t) & (abs(w.index - t) <= 2)]
    if len(tetangga) == 0:
        return pd.Series({"median_tetangga": np.nan, "z_lokal": np.nan})
    skala = (w.diff().abs().median() / 0.6745) or np.nan
    return pd.Series({"median_tetangga": tetangga.median(), "z_lokal": (v - tetangga.median()) / skala if skala else np.nan})


c = cand.merge(raw[["kode", "wilayah_id"]].drop_duplicates(), on="wilayah_id", how="left")
c = pd.concat([c, c.apply(konteks, axis=1)], axis=1)
c["di_luar_0_100"] = ~c["nilai"].between(0, 100)
c["kunci_kembar"] = c["indeks_raw"].map(raw.duplicated(KUNCI, keep=False))


def salah_desimal(r):
    m = r["median_tetangga"]
    if not np.isfinite(m) or m <= 0:
        return False
    return any(np.isclose(r["nilai"] * f, m, rtol=0.2) for f in (10, 0.1)) and abs(r["nilai"] - m) / m > 3


c["dugaan_salah_desimal"] = c.apply(salah_desimal, axis=1)


def klasifikasi(r):
    kunci = (r["tahun"], r["periode"])
    if r["di_luar_0_100"]:
        return "galat_input", "nilai di luar 0-100", False
    if r["kunci_kembar"]:
        return "galat_input", "kunci kembar di berkas mentah", True
    if r["dugaan_salah_desimal"]:
        return "galat_input", "dugaan salah desimal (nilai x10 atau /10 mendekati median tetangga); verifikasi ke JSON", True
    if kunci in patahan_set:
        dugaan = stat.loc[(stat["tahun"] == kunci[0]) & (stat["periode"] == kunci[1]), "kategori"].iloc[0] == "patahan_dugaan"
        if dugaan:
            return "patahan_seri", f"perubahan serentak yang menetap di {kunci[0]} {kunci[1]}; tidak ada catatan patahan di metadata ILO, dugaan", True
        return "patahan_seri", f"perubahan serentak yang menetap di {kunci[0]} {kunci[1]} dan ILO mencatat patahan di tahun yang sama", False
    if kunci in serentak_set:
        ilo = bool(stat.loc[(stat["tahun"] == kunci[0]) & (stat["periode"] == kunci[1]), "catatan_ilo_patahan"].iloc[0])
        catatan = "; ILO mencatat patahan di tahun ini tetapi pola berbalik, tinjau manual" if ilo else ""
        return "outlier_asli", "guncangan serentak di hampir semua provinsi yang berbalik pada tahun berikutnya atau sebelumnya" + catatan, ilo
    if np.isfinite(r["z_lokal"]) and abs(r["z_lokal"]) >= Z_ROBUST:
        return "outlier_asli", "lompatan lokal terhadap deret provinsi sendiri; tidak serentak dan tidak ada bukti galat", True
    return "outlier_asli", "ekstrem lintas provinsi tetapi wajar terhadap deret provinsi sendiri", False


hasil = c.apply(lambda r: pd.Series(klasifikasi(r), index=["kategori", "bukti", "perlu_tinjau"]), axis=1)
c = pd.concat([c, hasil], axis=1)
assert c["kategori"].notna().all() and len(c) == JUMLAH_CALON

kolom_out = ["id_calon", "nama_wilayah", "wilayah_id", "tahun", "periode", "nilai", "kategori", "bukti", "perlu_tinjau",
             "median_tetangga", "z_lokal", "indeks_raw", "baris_csv", "berkas_raw"]
c[kolom_out].round(3).to_csv(PROFIL / "anomali_terklasifikasi_var543.csv", index=False)
stat.round(3).to_csv(PROFIL / "patahan_serentak_var543.csv", index=False)

print("\nKategori calon:")
print(c["kategori"].value_counts().to_string())
print("\nPerlu tinjau manual:", int(c["perlu_tinjau"].sum()))
print(c.loc[c["perlu_tinjau"], ["id_calon", "nama_wilayah", "tahun", "periode", "nilai", "median_tetangga", "kategori", "bukti"]].round(2).to_string(index=False)
      if c["perlu_tinjau"].any() else "  tidak ada")

# ============ 4. GRAFIK ============
fig, ax = plt.subplots(figsize=(12, 5))
for per, m in [("Tahunan", "o"), ("Februari", "s"), ("Agustus", "^")]:
    s = stat[stat["periode"] == per]
    if len(s):
        ax.plot(s["tahun"], s["median_selisih"], marker=m, ms=3, label=per)
for r in serentak.itertuples():
    ax.axvline(r.tahun, color="red" if r.kategori.startswith("patahan") else "orange", alpha=0.3)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Tahun")
ax.set_ylabel("Median selisih tahun-ke-tahun lintas provinsi (poin persen)")
ax.set_title("Perubahan serentak TPT var543 (merah: patahan; oranye: guncangan tanpa catatan patahan)")
ax.legend()
fig.tight_layout()
fig.savefig(GAMBAR / "patahan_serentak_var543.png", dpi=200)
plt.close(fig)

METRIK_13 = {"outlier_ditandai": int(len(c)), "patahan_ditandai": int(len(patahan_set)),
             "calon_outlier_asli": int((c["kategori"] == "outlier_asli").sum()),
             "calon_patahan": int((c["kategori"] == "patahan_seri").sum()),
             "calon_galat": int((c["kategori"] == "galat_input").sum())}
print("\nMetrik untuk metrics.json:", METRIK_13)

Duplikat: baris dengan kunci kembar = 0 | baris identik penuh = 0
Snapshot BPS di paket: ['2026-10-17']
Tahun patahan menurut catatan ILO: {1990: ['UNE_DEAP_SEX_AGE_RT'], 2008: ['SDG_0852_SEX_AGE_RT', 'UNE_DEAP_SEX_AGE_RT'], 2013: ['SDG_0852_SEX_AGE_RT', 'UNE_DEAP_SEX_AGE_RT'], 2021: ['SDG_0852_SEX_AGE_RT', 'UNE_DEAP_SEX_AGE_RT']}

Tahun-periode serentak:
 periode  tahun  n_prov  median_selisih  share_searah  z_robust  berbalik                      catatan_ilo_patahan                         kategori
Februari   2021      34            0.91          0.94      3.76     False SDG_0852_SEX_AGE_RT, UNE_DEAP_SEX_AGE_RT guncangan_serentak_bukan_patahan
 Tahunan   1994      26            1.46          1.00      3.50     False                                                            patahan_dugaan
 Tahunan   2001      25            1.93          0.92      4.78     False                                                            patahan_dugaan



Kategori calon:
kategori
outlier_asli    75
patahan_seri     2

Perlu tinjau manual: 17
id_calon     nama_wilayah  tahun  periode  nilai  median_tetangga     kategori                                                                                              bukti
    A002             ACEH   2005  Agustus  14.00            10.14 outlier_asli           lompatan lokal terhadap deret provinsi sendiri; tidak serentak dan tidak ada bukti galat
    A003   SUMATERA UTARA   2006 Februari  14.82            10.63 outlier_asli           lompatan lokal terhadap deret provinsi sendiri; tidak serentak dan tidak ada bukti galat
    A004   SUMATERA BARAT   2004  Tahunan  12.74            10.00 outlier_asli           lompatan lokal terhadap deret provinsi sendiri; tidak serentak dan tidak ada bukti galat
    A006   SUMATERA BARAT   2006 Februari  12.93            11.02 outlier_asli           lompatan lokal terhadap deret provinsi sendiri; tidak serentak dan tidak ada bukti galat
    A007             


Metrik untuk metrics.json: {'outlier_ditandai': 77, 'patahan_ditandai': 2, 'calon_outlier_asli': 75, 'calon_patahan': 2, 'calon_galat': 0}


In [3]:
# ============ 5. 17 CALON PERLU TINJAU VS NASIONAL ============
nas_s = {p_: g.set_index("tahun")["nilai"] for p_, g in raw[nasional].groupby("periode")}
out = []
for r in c[c["perlu_tinjau"]].itertuples():
    w = raw[(raw["wilayah_id"] == r.wilayah_id) & (raw["periode"] == r.periode)].set_index("tahun")["nilai"]
    gap = (w - nas_s[r.periode].reindex(w.index)).dropna()
    tet = gap[(gap.index != r.tahun) & (abs(gap.index - r.tahun) <= 2)]
    out.append({"id_calon": r.id_calon, "wilayah": r.nama_wilayah, "tahun": r.tahun, "periode": r.periode,
                "gap_calon": round(gap.get(r.tahun, float("nan")), 2),
                "gap_biasa": round(tet.median(), 2) if len(tet) else None})
tinjau = pd.DataFrame(out)
tinjau["lonjakan_gap"] = (tinjau["gap_calon"] - tinjau["gap_biasa"]).round(2)
tinjau = tinjau.sort_values("lonjakan_gap", ascending=False)
tinjau.to_csv(PROFIL / "tinjau_vs_nasional_var543.csv", index=False)
print("Calon perlu tinjau vs nasional (lonjakan_gap):")
print(tinjau.to_string(index=False))


Calon perlu tinjau vs nasional (lonjakan_gap):
id_calon          wilayah  tahun  periode  gap_calon  gap_biasa  lonjakan_gap
    A068        GORONTALO   2005  Agustus       2.80      -2.30          5.10
    A077     MALUKU UTARA   2005  Agustus       1.85      -3.22          5.07
    A007             RIAU   2004  Tahunan       5.39       0.79          4.60
    A060 SULAWESI SELATAN   2003  Tahunan       7.65       3.23          4.42
    A003   SUMATERA UTARA   2006 Februari       4.37       0.88          3.49
    A008             RIAU   2005 Februari       3.65       0.83          2.82
    A072           MALUKU   2006 Februari       5.31       2.59          2.72
    A063 SULAWESI SELATAN   2005  Agustus       4.69       2.31          2.38
    A002             ACEH   2005  Agustus       2.76       0.44          2.32
    A004   SUMATERA BARAT   2004  Tahunan       2.88       0.63          2.25
    A074           MALUKU   2007 Februari       4.62       2.42          2.20
    A039         

In [4]:
# ============ 6. UJI SILANG CSV vs JSON ============
import json
jsn = json.load(open(RAW / "bps" / "2026-10-17" / "json" / "sdg8_tpt.json", encoding="utf-8"))
baris_j = []
for x in jsn:
    turvar = x["turvar"][0]["val"]
    for v in x["vervar"]:
        for t in x["tahun"]:
            for tt in x["turtahun"]:
                key = f'{v["val"]}543{turvar}{t["val"]}{tt["val"]}'
                if key in x["datacontent"]:
                    baris_j.append({"nama": str(v["label"]).strip().upper(), "tahun": int(t["label"]),
                                    "periode": tt["label"], "nilai_json": x["datacontent"][key]})
js = pd.DataFrame(baris_j)
cek = raw.assign(nama_u=raw["nama"].str.upper())[["nama_u", "tahun", "periode", "nilai"]].rename(columns={"nama_u": "nama"})
m = cek.merge(js, on=["nama", "tahun", "periode"], how="outer", indicator=True)
n_both = int((m["_merge"] == "both").sum())
n_beda = int(((m["_merge"] == "both") & ((m["nilai"] - m["nilai_json"]).abs() > 1e-9)).sum())
print("Uji silang CSV-JSON:", m["_merge"].value_counts().to_dict(), "| nilai berbeda:", n_beda)
assert n_both == len(raw) and n_beda == 0, "CSV dan JSON tidak sama; jangan tulis sama di log"


Uji silang CSV-JSON: {'both': 2000, 'left_only': 0, 'right_only': 0} | nilai berbeda: 0
